# Fire Hazard Experiment - Environment Check
Run this notebook in Google Colab before starting YOLO26 or YOLO26-Depth training. It checks Drive paths, GPU, packages, YOLO availability, dataset structure, annotation validity, previews, output writing, and final readiness.

## 1. Project configuration
Edit only this path cell if your Drive folders differ.

In [ ]:
DATASET_ROOT = '/content/drive/MyDrive/Thesis/fire_hazard_dataset'
OUTPUT_ROOT = '/content/drive/MyDrive/Thesis/fire_hazard_experiment_outputs'
DATA_YAML = f'{DATASET_ROOT}/data.yaml'
EXPECTED_IMAGE_COUNT = 23
seed = 42
checks = {}
print('DATASET_ROOT:', DATASET_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('DATA_YAML:', DATA_YAML)

## 2. Mount Google Drive and verify dataset root

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DATASET_ROOT_PATH = Path(DATASET_ROOT)
OUTPUT_ROOT_PATH = Path(OUTPUT_ROOT)
DATA_YAML_PATH = Path(DATA_YAML)
if DATASET_ROOT_PATH.exists():
    print('Dataset directory found:', DATASET_ROOT)
    checks['dataset_root'] = True
else:
    print('Dataset directory not found.\n')
    print('Expected:')
    print(DATASET_ROOT)
    print('\nPlease check DATASET_ROOT.')
    checks['dataset_root'] = False

## 3. Check Colab GPU environment

In [ ]:
import platform, sys
print('Python:', sys.version)
print('Operating system:', platform.platform())
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    print('CUDA version:', torch.version.cuda)
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print('GPU memory GB:', round(props.total_memory / (1024 ** 3), 2))
        checks['gpu'] = True
    else:
        print('\nWARNING: CUDA GPU is not available.\n')
        print('Before training:')
        print('Runtime -> Change runtime type -> GPU')
        checks['gpu'] = False
except Exception as exc:
    print('PyTorch check failed:', exc)
    checks['gpu'] = False

## 4. Install/check required packages

In [ ]:
import importlib, subprocess, sys
packages = {'ultralytics':'ultralytics','numpy':'numpy','pandas':'pandas','matplotlib':'matplotlib','Pillow':'PIL','PyYAML':'yaml','opencv-python-headless':'cv2'}
missing = []
for package, import_name in packages.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        missing.append(package)
if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('All required packages are already importable.')
import ultralytics, numpy as np, pandas as pd
print('Ultralytics version:', ultralytics.__version__)
print('NumPy version:', np.__version__)
print('Pandas version:', pd.__version__)
checks['packages'] = True

## 5. Verify YOLO26 and YOLO26-Depth availability

In [ ]:
from pathlib import Path
import ultralytics
root = Path(ultralytics.__file__).resolve().parent
def package_mentions(term):
    hits = []
    for path in root.rglob('*.py'):
        try:
            text = path.read_text(encoding='utf-8', errors='ignore')
        except Exception:
            continue
        if term.lower() in text.lower():
            hits.append(str(path.relative_to(root)))
    return hits[:25]
candidate_detection_models = ['yolo26n.pt','yolo26s.pt','yolo26m.pt','yolo26l.pt','yolo26x.pt']
candidate_depth_models = ['yolo26n-depth.pt','yolo26-depth-n.pt','yolo26n_depth.pt']
mentions_yolo26 = package_mentions('yolo26')
mentions_depth = package_mentions('depth')
checks['yolo26_detection'] = bool(mentions_yolo26)
checks['yolo26_depth'] = bool(mentions_yolo26) and bool(mentions_depth)
print('YOLO26 detection:')
print('Available:', 'YES' if checks['yolo26_detection'] else 'NO / not found in installed package text')
print('Candidate model names checked:', candidate_detection_models)
print('Package files mentioning yolo26:', mentions_yolo26 or 'none')
print('\nYOLO26-Depth:')
print('Available:', 'YES' if checks['yolo26_depth'] else 'NO / requires official API verification')
print('Candidate model names checked:', candidate_depth_models)
print('Package files mentioning depth:', mentions_depth or 'none')
print('Depth output: metric/relative requires runtime API or official documentation verification; do not infer from visualization colors.')

## 6. Inspect dataset structure and data.yaml

In [ ]:
import yaml
def print_tree(root, max_depth=4, max_entries=200):
    root = Path(root)
    print(root.name + '/')
    count = 0
    for path in sorted(root.rglob('*')):
        rel = path.relative_to(root)
        depth = len(rel.parts)
        if depth > max_depth:
            continue
        count += 1
        if count > max_entries:
            print('... output truncated ...')
            break
        print('    ' * (depth - 1) + '+-- ' + path.name + ('/' if path.is_dir() else ''))
if DATASET_ROOT_PATH.exists():
    print_tree(DATASET_ROOT_PATH)
else:
    print('Dataset root is missing, so structure inspection is skipped.')
print('\ndata.yaml contents:')
if DATA_YAML_PATH.exists():
    text = DATA_YAML_PATH.read_text(encoding='utf-8')
    print(text)
else:
    print('data.yaml not found:', DATA_YAML)
    checks['data_yaml'] = False

## 7. Validate data.yaml, image/label matching, and YOLO annotations

In [ ]:
import json, subprocess, sys
possible_roots = [Path.cwd(), Path.cwd().parent, Path('/content/fire-hazard-spatial'), Path('/content/drive/MyDrive/fire-hazard-spatial')]
REPO_ROOT = next((p for p in possible_roots if (p / 'scripts' / 'check_dataset.py').exists()), None)
if REPO_ROOT is None:
    print('Could not find scripts/check_dataset.py. Open this notebook from the cloned repository root.')
    checks['repository'] = False
    dataset_report = {'status': 'INVALID', 'problems': ['scripts/check_dataset.py not found']}
else:
    checks['repository'] = True
    report_json = Path('/tmp/fire_hazard_dataset_report.json')
    cmd = [sys.executable, str(REPO_ROOT / 'scripts' / 'check_dataset.py'), '--dataset-root', DATASET_ROOT, '--data-yaml', DATA_YAML, '--expected-images', str(EXPECTED_IMAGE_COUNT), '--json-output', str(report_json)]
    result = subprocess.run(cmd, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print('STDERR:', result.stderr)
    dataset_report = json.loads(report_json.read_text(encoding='utf-8')) if report_json.exists() else {'status': 'INVALID'}
    checks['data_yaml'] = bool(dataset_report.get('yaml_checks', {}).get('train')) and bool(dataset_report.get('yaml_checks', {}).get('val'))
    checks['annotations'] = dataset_report.get('status') == 'VALID'

## 8. Test writing to Google Drive

In [ ]:
ENV_CHECK_ROOT = OUTPUT_ROOT_PATH / 'environment_check'
try:
    ENV_CHECK_ROOT.mkdir(parents=True, exist_ok=True)
    test_file = ENV_CHECK_ROOT / 'environment_test.txt'
    expected = 'Google Drive output write test successful.'
    test_file.write_text(expected, encoding='utf-8')
    checks['drive_output'] = test_file.read_text(encoding='utf-8') == expected
    print('Google Drive output write test:', 'PASS' if checks['drive_output'] else 'FAIL')
except Exception as exc:
    checks['drive_output'] = False
    print('Google Drive output write test: FAIL', exc)

## 9. Visual annotation check

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
preview_dir = ENV_CHECK_ROOT / 'annotation_preview'
preview_dir.mkdir(parents=True, exist_ok=True)
class_names = {int(k): v for k, v in dataset_report.get('classes', {}).items()} if isinstance(dataset_report, dict) else {}
image_exts = {'.jpg','.jpeg','.png','.bmp','.webp','.tif','.tiff'}
all_images = sorted([p for p in DATASET_ROOT_PATH.rglob('*') if p.suffix.lower() in image_exts]) if DATASET_ROOT_PATH.exists() else []
random.seed(seed)
sample_images = random.sample(all_images, min(6, len(all_images))) if all_images else []
def label_for_image(image_path):
    parts = list(image_path.parts)
    for i, part in enumerate(parts):
        if part == 'images':
            parts[i] = 'labels'
            return Path(*parts).with_suffix('.txt')
    return image_path.with_suffix('.txt')
for idx, image_path in enumerate(sample_images, 1):
    im = Image.open(image_path).convert('RGB')
    w, h = im.size
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(im); ax.axis('off')
    label_path = label_for_image(image_path)
    if label_path.exists():
        for line in label_path.read_text(encoding='utf-8', errors='replace').splitlines():
            vals = line.split()
            if len(vals) != 5: continue
            cid, x, y, bw, bh = map(float, vals)
            x1, y1 = (x - bw / 2) * w, (y - bh / 2) * h
            ax.add_patch(patches.Rectangle((x1, y1), bw * w, bh * h, fill=False, linewidth=2, edgecolor='lime'))
            ax.text(x1, max(0, y1 - 4), class_names.get(int(cid), str(int(cid))), color='black', backgroundcolor='lime', fontsize=9)
    out = preview_dir / f'preview_{idx:02d}.png'
    fig.savefig(out, bbox_inches='tight', dpi=150)
    plt.show(); plt.close(fig)
checks['visual_preview'] = bool(sample_images)
print('Saved previews to:', preview_dir)

## 10. Optional one-image YOLO26 detection inference test

In [ ]:
RUN_DETECTION_SMOKE_TEST = False
DETECTION_MODEL_NAME = 'yolo26n.pt'
detection_inference_status = 'NOT RUN'
if RUN_DETECTION_SMOKE_TEST:
    try:
        from ultralytics import YOLO
        out = ENV_CHECK_ROOT / 'yolo26_test'; out.mkdir(parents=True, exist_ok=True)
        model = YOLO(DETECTION_MODEL_NAME)
        result = model.predict(str(sample_images[0]), save=True, project=str(out), name='predict', exist_ok=True)[0]
        print('Boxes available:', hasattr(result, 'boxes'))
        if hasattr(result, 'boxes') and result.boxes is not None: print('Number of boxes:', len(result.boxes))
        detection_inference_status = 'PASS'
    except Exception as exc:
        detection_inference_status = 'FAIL'; print('Detection smoke test failed:', exc)
else:
    print('Detection smoke test not run. Set RUN_DETECTION_SMOKE_TEST=True only after confirming YOLO26 checkpoint syntax.')
checks['detection_inference'] = detection_inference_status

## 11. Optional one-image YOLO26-Depth inference test

In [ ]:
RUN_DEPTH_SMOKE_TEST = False
DEPTH_MODEL_NAME = 'yolo26n-depth.pt'
depth_inference_status = 'NOT RUN'
if RUN_DEPTH_SMOKE_TEST:
    try:
        import numpy as np
        from ultralytics import YOLO
        out = ENV_CHECK_ROOT / 'depth_test'; out.mkdir(parents=True, exist_ok=True)
        result = YOLO(DEPTH_MODEL_NAME).predict(str(sample_images[0]))[0]
        depth = None
        for attr in ('depth','depths','pred_depth','maps'):
            if hasattr(result, attr) and getattr(result, attr) is not None:
                depth = getattr(result, attr); break
        if depth is None: raise RuntimeError('No numerical depth output attribute found on result object.')
        if hasattr(depth, 'data') and not isinstance(depth, np.ndarray): depth = depth.data
        if hasattr(depth, 'detach'): depth = depth.detach().cpu().numpy()
        depth = np.asarray(depth).squeeze()
        np.save(out / 'test_depth.npy', depth)
        plt.imshow(depth, cmap='magma'); plt.colorbar(label='model depth output'); plt.savefig(out / 'test_depth_visualization.png', bbox_inches='tight', dpi=150); plt.show()
        print('shape:', depth.shape, 'min:', float(np.nanmin(depth)), 'max:', float(np.nanmax(depth)), 'mean:', float(np.nanmean(depth)), 'median:', float(np.nanmedian(depth)))
        print('Output units/type: requires model documentation; not inferred from visualization colors.')
        depth_inference_status = 'PASS'
    except Exception as exc:
        depth_inference_status = 'FAIL'; print('Depth smoke test failed:', exc)
else:
    print('Depth smoke test not run. Set RUN_DEPTH_SMOKE_TEST=True only after confirming YOLO26-Depth support.')
checks['depth_inference'] = depth_inference_status

## 12. Generate environment report

In [ ]:
def pf(x): return 'PASS' if x else 'FAIL'
ready = all([checks.get('repository', True), checks.get('dataset_root', False), checks.get('data_yaml', False), checks.get('annotations', False), checks.get('gpu', False), checks.get('packages', False), checks.get('drive_output', False)])
lines = ['===========================================','FIRE HAZARD EXPERIMENT - ENVIRONMENT CHECK','===========================================','',f'Repository: {pf(checks.get("repository", True))}',f'Google Drive: {pf(Path("/content/drive").exists())}',f'Dataset root: {pf(checks.get("dataset_root", False))}',f'data.yaml: {pf(checks.get("data_yaml", False))}',f'Images: {dataset_report.get("total_images", "unknown")}',f'Labels: {dataset_report.get("total_labels", "unknown")}',f'Classes: {dataset_report.get("nc", "unknown")}',f'Annotation validation: {pf(checks.get("annotations", False))}',f'GPU: {pf(checks.get("gpu", False))}',f'PyTorch: {torch.__version__ if "torch" in globals() else "unknown"}',f'Ultralytics: {ultralytics.__version__}',f'YOLO26 detection: {pf(checks.get("yolo26_detection", False))}',f'YOLO26-Depth: {pf(checks.get("yolo26_depth", False))}',f'Google Drive output: {pf(checks.get("drive_output", False))}',f'Single-image detection inference: {checks.get("detection_inference", "NOT RUN")}',f'Single-image depth inference: {checks.get("depth_inference", "NOT RUN")}','']
if not ready:
    lines += ['Fix before full experiment:']
    if not checks.get('dataset_root', False): lines.append('- Correct DATASET_ROOT or upload the dataset to the expected Google Drive folder.')
    if not checks.get('data_yaml', False): lines.append('- Fix data.yaml paths/classes after reviewing the validator output.')
    if not checks.get('annotations', False): lines.append('- Fix missing labels, orphan labels, unreadable images, or malformed YOLO annotations listed above.')
    if not checks.get('gpu', False): lines.append('- Enable GPU: Runtime -> Change runtime type -> GPU.')
    if not checks.get('drive_output', False): lines.append('- Check OUTPUT_ROOT permissions and Google Drive storage.')
    lines.append('')
lines += [f'READY FOR FULL EXPERIMENT: {"YES" if ready else "NO"}', '===========================================']
final_report = '\n'.join(lines)
print(final_report)
ENV_CHECK_ROOT.mkdir(parents=True, exist_ok=True)
(ENV_CHECK_ROOT / 'environment_report.txt').write_text(final_report, encoding='utf-8')